# GEO-TEMPORAL ANALYSIS: OFFICE/WORK ZONES

In [ ]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as f
import findspark
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
import os
from pathlib import Path
import geopandas as gpd
from shapely import wkt
import json

In [ ]:
projectRoot = Path.cwd().parents[1]

CLEAN_DATA = projectRoot / "data" / "clean" / "clean_tlc_trip_record" / "fhvhv"
GEO_DATA = projectRoot / "data" / "NYC_Taxi_Zones_20251203.csv"
# csv files of plot data
TABLES_DIR = projectRoot / "data" / "plotTables" / "geo-temporal" / "fhvhv"
HTML_DIR = projectRoot / "data" / "html" / "geo_temporal" / "fhvhv"
os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(HTML_DIR, exist_ok=True)

In [ ]:
CLEAN_DATA

In [ ]:
def save_table(df:pd.DataFrame, name:str):
    path = os.path.join(str(TABLES_DIR), f"{name}.csv")
    df.to_csv(path, index=False)

def load_csv(path:Path) -> pd.DataFrame:
    return pd.read_csv(path)

def make_date_col(df):
    """
    Build a pandas datetime column from pickup_year + pickup_month.

    2020 01 -> 2020-01-01
    """
    df["date"] = pd.to_datetime(
        df["pickup_year"].astype(str) + "-" +
        df["pickup_month"].astype(str) + "-01"
    )
    return df

def add_temporal_columns(df: DataFrame) -> DataFrame:
    """
    Derive useful temporal features.
    """
    df = (
        df
        .withColumn("pickup_hour", f.hour("pickup_datetime"))
        .withColumn("pickup_day_of_week", f.dayofweek("pickup_datetime"))
        .withColumn("pickup_day_name", f.date_format("pickup_datetime", "EEEE"))
        .withColumn("pickup_is_weekend", f.dayofweek("pickup_datetime").isin([1, 7]))
        .withColumn("dropoff_year", f.year("dropoff_datetime"))
        .withColumn("dropoff_month", f.month("dropoff_datetime"))
        .withColumn("dropoff_hour", f.hour("dropoff_datetime"))
        .withColumn("dropoffp_day_of_week", f.dayofweek("dropoff_datetime"))
        .withColumn("dropoff_day_name", f.date_format("dropoff_datetime", "EEEE"))
        .withColumn("dropoff_is_weekend", f.dayofweek("dropoff_datetime").isin([1, 7]))
        .withColumn("time_of_day",
            f.when((f.col("pickup_hour") >= 6)  & (f.col("pickup_hour") < 10), "morning_rush")
             .when((f.col("pickup_hour") >= 10) & (f.col("pickup_hour") < 16), "midday")
             .when((f.col("pickup_hour") >= 16) & (f.col("pickup_hour") < 20), "evening_rush")
             .when((f.col("pickup_hour") >= 20) & (f.col("pickup_hour") < 24), "night")
             .otherwise("late_night")
        )
    )
    return df

In [ ]:
LAYOUT_DEFAULTS = dict(
    template="plotly_white",
    font=dict(family="Inter, sans-serif", size=12),
    hovermode="x unified",
    margin=dict(l=60, r=30, t=60, b=50),
)

DAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday",
             "Friday", "Saturday", "Sunday"]

# Spark Init

## No need to run Spark if you run the code from *.csv

`NOTE`:
Change the configs according to your device if you wanna run it (if you don't have the plot tables already in your device).

`If we want to run this code on the cloud,
we will need to change the configs too.`

In [ ]:
findspark.init()

spark = (SparkSession.builder
        .master("local[6]")  
        .config("spark.driver.memory", "8g")
        .config("spark.executor.memory", "8g")
        .config("spark.sql.shuffle.partitions", "100")
        .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "64m")
        .config("spark.driver.maxResultSize", "1g")
        .config("spark.sql.session.timeZone", "America/New_York")
        .config("spark.sql.timestampType", "TIMESTAMP_NTZ")
        # Network binding fixes for macOS
        .config("spark.driver.host", "127.0.0.1")
        .config("spark.driver.bindAddress", "127.0.0.1")
        .getOrCreate())

In [ ]:
spark.sparkContext

In [ ]:
df = spark.read.parquet(str(CLEAN_DATA))
df = add_temporal_columns(df)

In [ ]:
df_temp = pd.read_csv(GEO_DATA)

df_temp = df_temp.rename(columns ={"Shape Geometry" : "geometry", "Shape Length" : "length", "Shape Area" : "area", "Location ID" : "id", "Zone": "zone", "Borough" : "borough"})
    # Convierte WKT → geometría
df_temp['geometry'] = df_temp['geometry'].apply(wkt.loads) 
gdf = gpd.GeoDataFrame(df_temp, crs='EPSG:4326')

gdf_proj = gdf.to_crs('EPSG:2263')
gdf_proj['centroid'] = gdf_proj.geometry.centroid
gdf['lat'] = gdf_proj.centroid.to_crs('EPSG:4326').y
gdf['lon'] = gdf_proj.centroid.to_crs('EPSG:4326').x

# General Geo-Temporal Analysis

## Demand per hour and zones between 2023 and 2025

In [ ]:
def geo_temp_00_per_zones(df: DataFrame, gdf: DataFrame, option:str, table_path: Path):

    # 1. Load or compute aggregated data
    if not table_path or not table_path.exists():
        df_map = (
            df.filter(
                (f.col(f'{option}_year').between(2023, 2025))
            )
            .groupBy(f'{option}_hour', f'{option}_zone')
            .agg(f.count('*').alias('trips_per_hour'))
            .sort(f'{option}_hour')
            .toPandas()
        )
        save_table(df_map, f"geo_temp_00_per_zones_{option}")
    else:
        df_map = load_csv(table_path)

    # 2. Dissolve zones into borough-level polygons
    gdf_borough = gdf.dissolve(by="zone").reset_index()
    geojson = json.loads(gdf_borough.to_json())

    # 3. Choropleth — locations must match a GeoJSON property
    fig = px.choropleth_map(
        df_map,
        geojson=geojson,
        locations=f"{option}_zone",              # column in df_map
        featureidkey="properties.zone",       # path in GeoJSON features
        color="trips_per_hour",
        animation_frame=f"{option}_hour",
        color_continuous_scale="YlOrRd",
        map_style="carto-positron",
        zoom=10,
        center={"lat": 40.7128, "lon": -74.0060},
        range_color=[0, 1_000_000],
        opacity=0.7,
        title=f"Trips density per zones and hours",
    )

    fig.update_layout(
        margin={"r": 0, "t": 30, "l": 0, "b": 0},
        width=1200,
        height=800,
    )

    fig.show()
    fig.write_html(HTML_DIR / f"geo_temp_00_per_zones_{option}_hour.html")

In [ ]:
geo_temp_00_per_zones(df, gdf, 'pickup', None)

# geo_temp_00_per_zones(None, gdf, 'pickup', TABLES_DIR / "geo_temp_00_per_zones_pickup.csv")

In [ ]:
geo_temp_00_per_zones(df, gdf, 'dropoff', None)

# geo_temp_00_per_zones(None, gdf, 'dropoff', TABLES_DIR / "geo_temp_00_per_zones_dropoff.csv")

## Demanda per hours and boroughs between 2023 and 2025

In [ ]:
def geo_temp_08_per_borough(df: DataFrame, gdf: DataFrame, option:str, table_path: Path):

    # 1. Load or compute aggregated data
    if not table_path or not table_path.exists():
        df_map = (
            df.filter(
                (f.col(f'{option}_year').between(2023, 2025))
            )
            .groupBy(f'{option}_hour', f'{option}_borough')
            .agg(f.count('*').alias('trips_per_hour'))
            .sort(f'{option}_hour')
            .toPandas()
        )
        save_table(df_map, f"geo_temp_08_per_borough_{option}")
    else:
        df_map = load_csv(table_path)

    # 2. Dissolve zones into borough-level polygons
    gdf_borough = gdf.dissolve(by="borough").reset_index()
    geojson = json.loads(gdf_borough.to_json())

    # 3. Choropleth — locations must match a GeoJSON property
    fig = px.choropleth_map(
        df_map,
        geojson=geojson,
        locations=f"{option}_borough",              # column in df_map
        featureidkey="properties.borough",       # path in GeoJSON features
        color="trips_per_hour",
        animation_frame=f"{option}_hour",
        color_continuous_scale="YlOrRd",
        map_style="carto-positron",
        zoom=10,
        center={"lat": 40.7128, "lon": -74.0060},
        range_color=[0, 15_000_000],
        opacity=0.7,
        title=f"Trips density per borough and hours",
    )

    fig.update_layout(
        margin={"r": 0, "t": 30, "l": 0, "b": 0},
        width=1200,
        height=800,
    )

    fig.show()
    fig.write_html(HTML_DIR / f"geo_temp_08_per_borough_{option}_hour.html")

In [ ]:
geo_temp_08_per_borough(df, gdf, 'pickup', None)

# geo_temp_08_per_borough(None, gdf, 'pickup', TABLES_DIR / "geo_temp_08_per_borough_pickup.csv")

# General Analysis: Zones Year Month

## Demand per years, months and zones

In [ ]:
def geo_temp_01_zones_year_month(df: DataFrame, gdf: DataFrame, option:str, table_path: Path):

    # 1. Load or compute aggregated data
    if not table_path or not table_path.exists():
        df_map = (
            df.filter(
                (f.col(f'{option}_year').between(2023, 2025))
            )
            .groupBy(f'{option}_year', f'{option}_month', f'{option}_zone')
            .agg(f.count('*').alias('trips_per_year_month'))
            .sort(f'{option}_year', f'{option}_month')
            .toPandas()
        )
        save_table(df_map, f"geo_temp_01_zones_year_month_{option}")
    else:
        df_map = load_csv(table_path)
    df_map['year_month'] = df_map[f'{option}_year'].astype(str) + '-' + df_map[f'{option}_month'].astype(str).str.zfill(2)
    # 2. Dissolve zones into borough-level polygons
    gdf_borough = gdf.dissolve(by="zone").reset_index()
    geojson = json.loads(gdf_borough.to_json())

    # 3. Choropleth — locations must match a GeoJSON property
    fig = px.choropleth_map(
        df_map,
        geojson=geojson,
        locations=f"{option}_zone",              # column in df_map
        featureidkey="properties.zone",       # path in GeoJSON features
        color="trips_per_year_month",
        animation_frame="year_month",
        color_continuous_scale="YlOrRd",
        map_style="carto-positron",
        zoom=10,
        center={"lat": 40.7128, "lon": -74.0060},
        range_color=[0, 50_000],
        opacity=0.7,
        title=f"Trips density per zones, month and year",
    )

    fig.update_layout(
        margin={"r": 0, "t": 30, "l": 0, "b": 0},
        width=1200,
        height=800,
    )
    
    fig.show()
    fig.write_html(HTML_DIR / f"geo_temp_01_zones_year_month_{option}.html")

In [ ]:
geo_temp_01_zones_year_month(df, gdf, 'pickup', None)

# geo_temp_01_zones_year_month(None, gdf, 'pickup', TABLES_DIR / "geo_temp_01_zones_year_month_pickup.csv")

In [ ]:
geo_temp_01_zones_year_month(df, gdf, 'dropoff', None)

# geo_temp_01_zones_year_month(None, gdf, 'dropoff', TABLES_DIR / "geo_temp_01_zones_year_month_dropoff.csv")

## Demand per years and zones

In [ ]:
def geo_temp_06_zones_year(df: DataFrame, gdf: DataFrame, option:str, table_path: Path):

    # 1. Load or compute aggregated data
    if not table_path or not table_path.exists():
        df_map = (
            df.filter(
                (f.col(f'{option}_year').between(2023, 2025))
            )
            .groupBy(f'{option}_year', f'{option}_zone')
            .agg(f.count('*').alias('trips_per_year'))
            .sort(f'{option}_year')
            .toPandas()
        )
        save_table(df_map, f"geo_temp_06_zones_year_{option}")
    else:
        df_map = load_csv(table_path)
    # 2. Dissolve zones into borough-level polygons
    gdf_borough = gdf.dissolve(by="zone").reset_index()
    geojson = json.loads(gdf_borough.to_json())

    # 3. Choropleth — locations must match a GeoJSON property
    fig = px.choropleth_map(
        df_map,
        geojson=geojson,
        locations=f"{option}_zone",              # column in df_map
        featureidkey="properties.zone",       # path in GeoJSON features
        color="trips_per_year",
        animation_frame=f"{option}_year",
        color_continuous_scale="YlOrRd",
        map_style="carto-positron",
        zoom=10,
        center={"lat": 40.7128, "lon": -74.0060},
        range_color=[0, 5_000_000],
        opacity=0.7,
        title=f"Trips density per zones and year",
    )

    fig.update_layout(
        margin={"r": 0, "t": 30, "l": 0, "b": 0},
        width=1200,
        height=800,
    )
    
    fig.show()
    fig.write_html(HTML_DIR / f"geo_temp_06_zones_year_{option}.html")

In [ ]:
geo_temp_06_zones_year(df, gdf, 'pickup', None)
#geo_temp_06_zones_year(None, gdf, 'pickup', TABLES_DIR / "geo_temp_06_zones_year_pickup.csv")

## Demand per years and boroughs

In [ ]:
def geo_temp_07_borough_year(df: DataFrame, gdf: DataFrame, option:str, table_path: Path):

    # 1. Load or compute aggregated data
    if not table_path or not table_path.exists():
        df_map = (
            df.filter(
                (f.col(f'{option}_year').between(2023, 2025))
            )
            .groupBy(f'{option}_year', f'{option}_borough')
            .agg(f.count('*').alias('trips_per_year'))
            .sort(f'{option}_year')
            .toPandas()
        )
        save_table(df_map, f"geo_temp_07_borough_year_{option}")
    else:
        df_map = load_csv(table_path)
    # 2. Dissolve zones into borough-level polygons
    gdf_borough = gdf.dissolve(by="borough").reset_index()
    geojson = json.loads(gdf_borough.to_json())

    # 3. Choropleth — locations must match a GeoJSON property
    fig = px.choropleth_map(
        df_map,
        geojson=geojson,
        locations=f"{option}_borough",              # column in df_map
        featureidkey="properties.borough",       # path in GeoJSON features
        color="trips_per_year",
        animation_frame=f"{option}_year",
        color_continuous_scale="YlOrRd",
        map_style="carto-positron",
        zoom=10,
        center={"lat": 40.7128, "lon": -74.0060},
        range_color=[0, 100_000_000],
        opacity=0.7,
        title=f"Trips density per borough and year",
    )

    fig.update_layout(
        margin={"r": 0, "t": 30, "l": 0, "b": 0},
        width=1200,
        height=800,
    )
    
    fig.show()
    fig.write_html(HTML_DIR / f"geo_temp_07_borough_year_{option}.html")

In [ ]:
geo_temp_07_borough_year(df, gdf, 'pickup', None)
#geo_temp_07_borough_year(None, gdf, 'pickup', TABLES_DIR / "geo_temp_07_borough_year_pickup.csv")

# Filtering: Only L-V from 6 a.m to 8 p.m

In [ ]:
df_work_time = (
    df.filter(
        (f.col('pickup_year').between(2023, 2025)) &
        (f.col('pickup_is_weekend') == 'false') &
        (f.col('pickup_hour').between(6, 20)) &
        (~f.col('PULocationID').isin([1, 132, 138])) &
        (~f.col('DOLocationID').isin([1, 132, 138]))
    )
)

In [ ]:
df_work_time.count()

# General Plot: All zones

In [ ]:
def geo_temp_02_per_borough_work_time(df: DataFrame, gdf: DataFrame, table_path: Path):

    # 1. Load or compute aggregated data
    if not table_path or not table_path.exists():
        # Primero calculamos los viajes por hora y borough
        hourly_borough_counts = (
            df.groupBy('pickup_hour', 'pickup_borough')
            .agg(f.count('*').alias('trips_per_hour'))
        )
        
        # Calculamos el total de viajes por hora
        hourly_totals = (
            hourly_borough_counts.groupBy('pickup_hour')
            .agg(f.sum('trips_per_hour').alias('total_trips_per_hour'))
        )
        
        # Unimos y calculamos la proporción
        df_map = (
            hourly_borough_counts.join(hourly_totals, 'pickup_hour')
            .withColumn(
                'proportion', 
                f.col('trips_per_hour') / f.col('total_trips_per_hour')
            )
            .withColumn(
                'percentage', 
                (f.col('trips_per_hour') / f.col('total_trips_per_hour') * 100)
            )
            .select('pickup_hour', 'pickup_borough', 'proportion', 'percentage', 'trips_per_hour')
            .sort('pickup_hour')
            .toPandas()
        )
        save_table(df_map, "geo_temp_02_per_borough_work_time")
    else:
        df_map = load_csv(table_path)

    # 2. Dissolve zones into borough-level polygons
    gdf_borough = gdf.dissolve(by="borough").reset_index()
    geojson = json.loads(gdf_borough.to_json())

    # 3. Choropleth — ahora usando proporción o porcentaje
    fig = px.choropleth_map(
        df_map,
        geojson=geojson,
        locations="pickup_borough",
        featureidkey="properties.borough",
        color="proportion", 
        animation_frame="pickup_hour",
        color_continuous_scale="YlOrRd",
        range_color=[0, 1],
        map_style="carto-positron",
        zoom=9,
        center={"lat": 40.7128, "lon": -74.0060},
        opacity=0.7,
        title="Trips per borough and hours (pct)",  # Título actualizado
    )

    fig.update_layout(
        margin={"r": 0, "t": 30, "l": 0, "b": 0},
        width=1200,
        height=800,
    )

    fig.show()
    fig.write_html(HTML_DIR / "geo_temp_02_per_borough_work_time.html")

In [ ]:
geo_temp_02_per_borough_work_time(df_work_time, gdf, None)

# geo_temp_02_per_borough_work_time(None, gdf, TABLES_DIR / "geo_temp_02_per_borough_work_time.csv")

## Per Zones

In [ ]:
def geo_temp_03_per_work_time(df: DataFrame, gdf: DataFrame, option:str, table_path: Path):

    # 1. Load or compute aggregated data
    if not table_path or not table_path.exists():
        df_map = (
            df
            .groupBy(f'{option}_hour', f'{option}_zone')
            .agg(f.count('*').alias('trips_per_hour'))
            .sort(f'{option}_hour')
            .toPandas()
        )
        save_table(df_map, f"geo_temp_03_per_work_time_{option}")
    else:
        df_map = load_csv(table_path)

    # 2. Dissolve zones into borough-level polygons
    gdf_borough = gdf.dissolve(by="zone").reset_index()
    geojson = json.loads(gdf_borough.to_json())

    # 3. Choropleth — locations must match a GeoJSON property
    fig = px.choropleth_map(
        df_map,
        geojson=geojson,
        locations=f"{option}_zone",              # column in df_map
        featureidkey="properties.zone",       # path in GeoJSON features
        color="trips_per_hour",
        animation_frame=f"{option}_hour",
        color_continuous_scale="YlOrRd",
        map_style="carto-positron",
        zoom=9,
        center={"lat": 40.7128, "lon": -74.0060},
        range_color=[0, 400_000],
        opacity=0.7,
        title=f"Trips density per zones and hours {option}",
    )

    fig.update_layout(
        margin={"r": 0, "t": 30, "l": 0, "b": 0},
        width=1200,
        height=800,
    )

    fig.show()
    fig.write_html(HTML_DIR / f"geo_temp_03_per_work_time_{option}.html")

In [ ]:
geo_temp_03_per_work_time(df_work_time, gdf, 'pickup', None)

# geo_temp_03_per_work_time(None, gdf, 'pickup', TABLES_DIR / "geo_temp_03_per_work_time_pickup.csv")

In [ ]:
geo_temp_03_per_work_time(df_work_time, gdf, 'dropoff', None)

# geo_temp_03_per_work_time(None, gdf, 'dropoff', TABLES_DIR / "geo_temp_03_per_work_time_dropoff.csv")

# Weekends

In [ ]:
def generate_cloropleth_map(df_map: pd.DataFrame, geojson: json, option: str, name: str):
   
    title_borough = f"Densidad de viajes por hora y zonas"
    title_html = f"geo_temp_04_leisure_{name}_{option}.html"
    fig = px.choropleth_map(
        df_map,
        geojson=geojson,
        locations=f"{option}_zone",              # column in df_map
        featureidkey="properties.zone",       # path in GeoJSON features
        color="trips_per_hour",
        animation_frame="pickup_hour",
        color_continuous_scale="YlOrRd",
        map_style="carto-positron",
        zoom=9,
        center={"lat": 40.7128, "lon": -74.0060},
        opacity=0.7,
        title=title_borough
    )

    fig.update_layout(
        margin={"r": 0, "t": 30, "l": 0, "b": 0},
        width=1200,
        height=800,
    )

    fig.show()
    fig.write_html(HTML_DIR / title_html)

def geo_temp_04_leisure(df: DataFrame, gdf: DataFrame, name : str, table_path: Path = None, option: str = 'pickup'):
    if not table_path or not table_path.exists():
        df_map = (
            df.groupBy('pickup_hour', f'{option}_zone')
            .agg(f.count('*').alias('trips_per_hour'))
            .sort('pickup_hour')
            .toPandas()
        )
        save_table(df_map, f"geo_temp_04_leisure_{name}_{option}")
    else:
        df_map = load_csv(table_path)

    # Dissolve zones into borough-level polygons
    gdf_borough = gdf.dissolve(by="zone").reset_index()
    geojson = json.loads(gdf_borough.to_json())

    # Choropleth
    generate_cloropleth_map(df_map, geojson, option, name)

In [ ]:
df_weekend = (df.select([f.col("pickup_zone"), f.col("dropoff_zone"), f.col("pickup_hour"), 
                          f.col("time_of_day"), f.col("total_amount"), f.col("pickup_is_weekend"), f.col("pickup_year")]))

df_weekend2 = (df_weekend.filter((f.col("pickup_is_weekend") == True) & (~ f.col("pickup_zone").contains("Airport")) &
                        (~ f.col("dropoff_zone").contains("Airport"))
))

In [ ]:
geo_temp_04_leisure(df_weekend2, gdf,name = "weekend_fhv")

# The most expensive areas (when and where)

In [ ]:
def generate_cloropleth_map_cost(df_map: pd.DataFrame, geojson: json, option: str, name: str):

    title_borough = f"Coste del viaje por hora y zonas"
    title_html = f"geo_temp_05_cost_fhv_{name}_{option}.html"
    fig = px.choropleth_map(
        df_map,
        geojson=geojson,
        locations=f"{option}_zone",              # column in df_map
        featureidkey="properties.zone",       # path in GeoJSON features
        color="avg_cost",
        animation_frame="request_hour", # use request hour to understand when is more expensive to book a trip in a zone
        color_continuous_scale="YlOrRd",
        map_style="carto-positron",
        zoom=9,
        center={"lat": 40.7128, "lon": -74.0060},
        opacity=0.7,
        title=title_borough,
        hover_data=["pickup_zone","total_amount_hour", "n_trips", "avg_distance", "max_cost"]
    )

    fig.update_layout(
        margin={"r": 0, "t": 30, "l": 0, "b": 0},
        width=1200,
        height=800,
    )

    fig.show()
    fig.write_html(HTML_DIR / title_html)

def geo_temp_05_cost(df: DataFrame, gdf: DataFrame, name : str, table_path: Path = None, option: str = 'pickup'):
    if not table_path or not table_path.exists():
        df_map = (
            df.groupBy('request_hour', f'{option}_zone')
            .agg(f.sum("total_amount").alias("total_amount_hour"), f.count('*').alias('n_trips'), 
                 f.max("total_amount").alias("max_cost"),f.sum("trip_miles").alias("total_distance_hour"))
            .withColumn('avg_cost', f.col("total_amount_hour") / f.col('n_trips'))
            .withColumn('avg_distance', f.col("total_distance_hour") / f.col('n_trips'))
            .sort('request_hour')
            .toPandas()
        )
        save_table(df_map, f"geo_temp_05_cost_fhv{name}_{option}")
    else:
        df_map = load_csv(table_path)

    # 2. Dissolve zones into borough-level polygons
    gdf_borough = gdf.dissolve(by="zone").reset_index()
    geojson = json.loads(gdf_borough.to_json())

    # 3. Choropleth
    generate_cloropleth_map_cost(df_map, geojson, option, name)


In [ ]:
df.select("trip_miles").agg(f.max("trip_miles"), f.min("trip_miles"), f.avg("trip_miles"), f.mode("trip_miles")).show()

In [ ]:
#Originally, calculated it only with 2025
p25, p50, p75, p90 = df.approxQuantile("trip_miles", [0.25, 0.5, 0.75, 0.9], 0.01)
print(f"25th percentile: {p25}, 50th percentile: {p50}, 75th percentile: {p75}, 90th percentile: {p90}")

In [ ]:
df_money = (df.select([f.col("pickup_zone"), f.col("dropoff_zone"), f.col("pickup_hour"), 
                          f.col("trip_miles"), f.col("total_amount"), f.col("request_datetime")]))
df_money = (df_money.
            withColumn("lenght_trip", f.when(f.col("trip_miles") < 1.5, "short")
            .when((f.col("trip_miles") >= 1.5) & (f.col("trip_miles") < 6), "medium")
            .otherwise("long")))
df_money = df_money.withColumn("request_hour", f.hour("request_datetime"))

df_money_trip = (df_money.filter((f.col("trip_miles") < 10)))
 

In [ ]:
df_money_trip_short = df_money_trip.filter((f.col("lenght_trip") == "short") & (f.col("total_amount") < 50))
geo_temp_05_cost(df_money_trip_short, gdf, "shortDistance")

In [ ]:
df_money_trip_med = df_money_trip.filter((f.col("lenght_trip") == "medium") & (f.col("total_amount") < 100))
geo_temp_05_cost(df_money_trip_med, gdf, "mediumDistance")


In [ ]:
geo_temp_05_cost(df_money_trip.filter((f.col("lenght_trip") == "long")), gdf, "longDistance")

In [ ]:
gdf

In [ ]:
# fig = px.choropleth_map(
#     gdf, # gdf es un dataframe de GeoPandas
#     geojson=gdf.geometry, # Una columna con datos de forma MULTIPOLYGON
#     locations=gdf.index,   # Usamos el índice para alinear (0, 1, 2...)
#     color="borough",       # Colorear por 'borough' (Bronx, Queens...)
#     # O podrías usar color="area" para ver tamaño
#     hover_name="zone",     # Al pasar el ratón mostrará "Newark Airport", etc.
#     hover_data=["id", "area"], # Datos extra al pasar el ratón, se indica el nombre de las columnas
#     #map_style="carto-positron", # Fondo limpio
#     map_style="open-street-map",  # Este es el estilo clásico de Folium/OSM
#     opacity=0.6,
#     center={"lat": 40.7128, "lon": -74.0060}, # Centro de NYC
#     zoom=9
# )

# fig.update_layout(
#         margin={"r":0,"t":0,"l":0,"b":0}, 
#         width=1200,
#         height=800,)
# fig.show()

In [ ]:
#spark.stop()